# Метрики OCR: GOT-OCR2.0

Ноутбук для **Colab / локально**: прогон **`ucaslcl/GOT-OCR2_0`** (`model.chat`, легче, чем полный DeepSeek-OCR), расчёт **CER / Accuracy / Final Score** (как в `scripts/task03_compare_ocr_text_to_reference.py`) и сводная строка под таблицу метрик.

| Поле | Откуда в ноутбуке |
|--------|-------------------|
| **Model** | `MODEL_ID` + режим `OCR_TYPE` (`ocr` / `format`) |
| **Final Score** | `100 × (1 − CER)` по микро-агрегации по символам |
| **Accuracy** | `1 − CER` (посимвольно, после нормализации) |
| **CER** | `jiwer` на нормализованных строках |
| **Unit Test Rate** | доля файлов, прошедших порог CER **и** (если задан) JSON с ожиданиями |
| **Внутренний парсер** | текстовая метка (например `none` или имя своего пост-процессора) |
| **Токены** | длина гипотезы в токенах базового токенайзера модели + символы |
| **Доп. Комментарии** | агрегаты, устройство, ошибки загрузки |

**Данные:** положите PNG в `input/data/1/` (или укажите `INPUT_DIR`). Эталон для CER — файл рядом: `имя.ref.txt` или `имя.txt` (тот же stem, что у изображения).

**Без эталона:** CER в этом прогоне **не считается**; после OCR в каталог изображения записывается черновик **`имя.ref.txt`** с текущим выводом модели — отредактируйте его и перезапустите ячейку прогона, чтобы появились CER и агрегаты.

Зависимости: `notes/requirements-ocr-notebook-colab.txt` (в т.ч. **verovio** для remote code GOT-OCR2).

**Пути:** в **Google Colab** сначала **`git clone`** репозитория и **`%cd`** в его корень (где лежат `notes/` и `scripts/`) — иначе `cwd` часто остаётся `/content` и пути к `input/` не сработают. Локально или в Codespaces, если ноутбук уже открыт из корня репо, clone не нужен. Дополнительно корень ищется функцией `find_repo_root()` (подпапки и родители `cwd`).

In [ ]:
# Google Colab: раскомментируйте и перейдите в корень клона (репозиторий с input/ и scripts/).
# !git clone https://github.com/developer-mixa/OCR-Analyze.git /content/OCR-Analyze
# %cd /content/OCR-Analyze

# Установка пакетов (из корня репозитория после clone + %cd)
import sys, subprocess

def pip_install(packages: list[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# GOT-OCR2.0: verovio обязателен для импорта remote code; torch/transformers — из среды Colab.
pip_install([
    "jiwer",
    "pandas",
    "tqdm",
    "Pillow",
    "safetensors",
    "accelerate",
    "sentencepiece",
    "protobuf",
    "verovio",
    "transformers>=4.40,<6",
])
print("OK: зависимости установлены.")
print("При смене transformers перезапустите kernel (Colab: Runtime → Restart session).")

In [ ]:
from __future__ import annotations

import json
import re
import time
import unicodedata
from pathlib import Path

import jiwer
import pandas as pd
from IPython.display import HTML, display
from tqdm.auto import tqdm

# Если авто-поиск не найдёт репо — задайте абсолютный путь вручную и перезапустите ячейку.
REPO_ROOT_OVERRIDE: Path | None = None


def find_repo_root() -> Path:
    """Каталог с notes/ и scripts/ — от cwd вверх и среди прямых подпапок cwd."""
    if REPO_ROOT_OVERRIDE is not None:
        p = REPO_ROOT_OVERRIDE.expanduser().resolve()
        if (p / "notes").is_dir() and (p / "scripts").is_dir():
            return p
        print("WARN: REPO_ROOT_OVERRIDE задан, но нет notes/ и scripts/ — игнорируем.")
    cwd = Path.cwd().resolve()
    starts: list[Path] = []
    if cwd.name == "notes":
        starts.append(cwd.parent)
    starts.append(cwd)
    try:
        subdirs = sorted([p for p in cwd.iterdir() if p.is_dir()], key=lambda x: x.name.lower())
        starts.extend(subdirs)
    except OSError:
        pass
    seen: set[Path] = set()
    ordered: list[Path] = []
    for s in starts:
        s = s.resolve()
        if s not in seen:
            seen.add(s)
            ordered.append(s)
    for start in ordered:
        for cand in [start, *start.parents]:
            try:
                if (cand / "notes").is_dir() and (cand / "scripts").is_dir():
                    return cand
            except OSError:
                continue
    return cwd


REPO_ROOT = find_repo_root()

INPUT_DIR = REPO_ROOT / "input" / "data" / "1"
OUTPUT_DIR = REPO_ROOT / "output" / "got_ocr_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Опционально: JSON с проверками вида {"test1.png": {"must_contain": ["##", "Таблица"]}}
UNIT_TESTS_JSON = REPO_ROOT / "input" / "data" / "1" / "ocr_unit_expectations.json"

MODEL_ID = "ucaslcl/GOT-OCR2_0"
# Режимы из README GOT-OCR2: "ocr" | "format" | при необходимости расширьте вызов в infer_one
OCR_TYPE = "ocr"

NORMALIZE_MODE = "nfkc_ws"  # как task03 по умолчанию
CER_PASS_THRESHOLD = 0.05
INTERNAL_PARSER_LABEL = "none"

print("REPO_ROOT =", REPO_ROOT.resolve())
print("INPUT_DIR =", INPUT_DIR.resolve(), "exists:", INPUT_DIR.is_dir())

In [ ]:
def normalize_text(s: str, mode: str) -> str:
    if mode == "none":
        return s
    t = s
    if mode in ("nfkc", "nfkc_ws", "nfkc_ws_lower"):
        t = unicodedata.normalize("NFKC", t)
    if mode in ("nfkc_ws", "nfkc_ws_lower", "ws", "ws_lower"):
        t = re.sub(r"\s+", " ", t).strip()
    if mode in ("nfkc_ws_lower", "ws_lower"):
        t = t.lower()
    return t


def char_error_rate(ref: str, hyp: str) -> float:
    if not ref and not hyp:
        return 0.0
    if not ref:
        return 1.0
    return float(jiwer.cer(ref, hyp))


def load_reference_for_image(img_path: Path) -> str | None:
    """Эталон: stem.ref.txt, затем stem.txt, stem.md. Пустой/пробельный файл = нет эталона."""
    stem = img_path.stem
    for name in (f"{stem}.ref.txt", f"{stem}.txt", f"{stem}.md"):
        p = img_path.parent / name
        if p.is_file():
            text = p.read_text(encoding="utf-8")
            if text.strip():
                return text
    return None


def load_unit_expectations(path: Path) -> dict:
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def run_unit_checks(hyp_raw: str, rules: dict | None) -> tuple[bool, list[str]]:
    if not rules:
        return True, []
    fails: list[str] = []
    for sub in rules.get("must_contain") or []:
        if sub not in hyp_raw:
            fails.append(f"missing substring: {sub!r}")
    for sub in rules.get("must_not_contain") or []:
        if sub in hyp_raw:
            fails.append(f"forbidden substring present: {sub!r}")
    return (len(fails) == 0), fails


def pick_torch_dtype():
    import torch

    if not torch.cuda.is_available():
        return torch.float32, "cpu"
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        return torch.bfloat16, "cuda"
    return torch.float16, "cuda"

In [ ]:
import torch
import transformers
from transformers import AutoModel, AutoTokenizer

print("transformers", transformers.__version__, "| torch", torch.__version__)
dtype, device_str = pick_torch_dtype()
print(f"device={device_str}, dtype={dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

load_kw = dict(
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    pad_token_id=tokenizer.eos_token_id,
)
if device_str == "cuda":
    load_kw["device_map"] = "cuda"

model = AutoModel.from_pretrained(MODEL_ID, **load_kw)
model.eval()
if device_str == "cpu":
    model = model.to(torch.float32)

print("OK: модель загружена.")

In [ ]:
def infer_one(image_path: Path) -> tuple[str, float]:
    t0 = time.perf_counter()
    res = model.chat(tokenizer, str(image_path), ocr_type=OCR_TYPE)
    elapsed = time.perf_counter() - t0
    hyp = res if isinstance(res, str) else str(res)
    return hyp, elapsed


def model_label() -> str:
    return f"{MODEL_ID} | chat ocr_type={OCR_TYPE} | low_cpu_mem_usage device_map=cuda(if cuda)"

## Прогон по изображениям

Если каталога `input/data/1` нет — создайте его и добавьте PNG и (по желанию) эталоны `*.ref.txt`.

Опционально — `input/data/1/ocr_unit_expectations.json` для **Unit Test Rate**:

```json
{ "page1.png": { "must_contain": ["#"] } }
```


In [7]:
expectations = load_unit_expectations(UNIT_TESTS_JSON)

image_paths = sorted(INPUT_DIR.glob("*.png")) if INPUT_DIR.is_dir() else []
if not image_paths:
    print("Нет PNG в", INPUT_DIR, "— добавьте файлы для теста.")

rows: list[dict] = []
corpus_ref_parts: list[str] = []
corpus_hyp_parts: list[str] = []

for img_path in tqdm(image_paths, desc="GOT-OCR2"):
    ref_raw = load_reference_for_image(img_path)
    had_ref = bool(ref_raw and ref_raw.strip())
    try:
        hyp_raw, sec = infer_one(img_path)
    except Exception as e:
        rows.append(
            {
                "file": img_path.name,
                "error": repr(e),
                "elapsed_sec": None,
                "CER": None,
                "char_accuracy": None,
                "ref_chars": len(normalize_text(ref_raw, NORMALIZE_MODE)) if had_ref else None,
                "hyp_chars": None,
                "hyp_tokens": None,
                "cer_pass": None,
                "unit_ok": None,
                "unit_fails": None,
                "ref_status": None,
            }
        )
        continue

    out_md = OUTPUT_DIR / f"{img_path.stem}_hypothesis.md"
    out_md.write_text(hyp_raw, encoding="utf-8")

    ref_status = "эталон из файла" if had_ref else None
    if not had_ref and hyp_raw.strip():
        draft_ref = img_path.parent / f"{img_path.stem}.ref.txt"
        draft_ref.write_text(hyp_raw, encoding="utf-8")
        ref_status = f"черновик эталона записан (правьте и перезапустите): {draft_ref.name}"
    elif not had_ref:
        ref_status = "нет эталона, вывод OCR пуст — черновик не создан"

    ref_n = normalize_text(ref_raw or "", NORMALIZE_MODE)
    hyp_n = normalize_text(hyp_raw, NORMALIZE_MODE)
    cer = char_error_rate(ref_n, hyp_n) if had_ref else None
    acc = max(0.0, min(1.0, 1.0 - cer)) if cer is not None else None

    if had_ref:
        corpus_ref_parts.append(ref_n)
        corpus_hyp_parts.append(hyp_n)

    tok_len = None
    try:
        tok_len = len(tokenizer.encode(hyp_raw, add_special_tokens=False))
    except Exception:
        pass

    rules = expectations.get(img_path.name)
    u_ok, u_fails = run_unit_checks(hyp_raw, rules)
    cer_pass = (cer is not None and cer <= CER_PASS_THRESHOLD) if had_ref else None

    combined_pass = None
    if had_ref:
        combined_pass = cer_pass and u_ok
    elif rules:
        combined_pass = u_ok

    rows.append(
        {
            "file": img_path.name,
            "error": None,
            "elapsed_sec": round(sec, 4),
            "CER": round(cer, 6) if cer is not None else None,
            "char_accuracy": round(acc, 6) if acc is not None else None,
            "ref_chars": len(ref_n) if had_ref else None,
            "ref_status": ref_status,
            "hyp_chars": len(hyp_n),
            "hyp_tokens": tok_len,
            "cer_pass": cer_pass,
            "unit_ok": u_ok,
            "unit_fails": "; ".join(u_fails) if u_fails else None,
            "combined_pass": combined_pass,
        }
    )

df = pd.DataFrame(rows)
display(HTML("<h3>По файлам</h3>"))
if not df.empty:
    display(df.style.hide(axis="index").format(
        {"elapsed_sec": "{:.4f}", "CER": "{:.6f}"},
        na_rep="—",
    ))
else:
    display(df)

GOT-OCR2:   0%|          | 0/4 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


file,error,elapsed_sec,CER,char_accuracy,ref_chars,ref_status,hyp_chars,hyp_tokens,cer_pass,unit_ok,unit_fails,combined_pass
test1.png,—,24.1191,0.000000,1.000000,1273,эталон из файла,1273,596,True,True,—,True
test2.png,—,46.2699,0.000000,1.000000,1304,эталон из файла,1304,1151,True,True,—,True
test3.png,—,46.2995,0.000000,1.000000,2375,эталон из файла,2375,1147,True,True,—,True
test4.png,—,17.5400,0.000000,1.000000,931,эталон из файла,931,432,True,True,—,True


In [8]:
# Микро-CER: одна пара больших строк (конкатенация с \n), эквивалентно сумме правок / длине эталона
micro_cer = (
    char_error_rate("\n".join(corpus_ref_parts), "\n".join(corpus_hyp_parts))
    if corpus_ref_parts
    else None
)
micro_acc = max(0.0, min(1.0, 1.0 - micro_cer)) if micro_cer is not None else None
final_score = round(100.0 * (1.0 - micro_cer), 2) if micro_cer is not None else None

evaluable = df["combined_pass"].notna() if not df.empty else pd.Series(dtype=bool)
unit_test_rate = float(df.loc[evaluable, "combined_pass"].mean()) if evaluable.any() else None

total_hyp_tokens = int(df["hyp_tokens"].fillna(0).sum()) if not df.empty else 0
mean_elapsed = float(df["elapsed_sec"].dropna().mean()) if not df.empty and df["elapsed_sec"].notna().any() else None

draft_ref_writes = 0
if not df.empty and "ref_status" in df.columns:
    draft_ref_writes = int(
        df["ref_status"].fillna("").str.contains("черновик эталона записан", regex=False).sum()
    )

comments = (
    f"model={MODEL_ID} ocr_type={OCR_TYPE}; device={device_str}; dtype={dtype}; "
    f"n_images={len(image_paths)}; with_ref_for_cer={df['CER'].notna().sum() if not df.empty else 0}; "
    f"draft_ref_txt_written={draft_ref_writes} (без эталона до прогона: CER не считался, записан черновик .ref.txt); "
    f"mean_elapsed_s={mean_elapsed}; "
    f"CER_PASS_THRESHOLD={CER_PASS_THRESHOLD}; normalize={NORMALIZE_MODE}; "
    f"expectations_file={UNIT_TESTS_JSON.name}"
)

summary_row = {
    "Model": model_label(),
    "Final Score": final_score,
    "Accuracy": round(micro_acc, 6) if micro_acc is not None else None,
    "CER": round(micro_cer, 6) if micro_cer is not None else None,
    "Unit Test Rate": round(unit_test_rate, 4) if unit_test_rate is not None else None,
    "Внутренний парсер": INTERNAL_PARSER_LABEL,
    "Токены": {
        "output_tokens_sum_hypothesis": total_hyp_tokens,
        "note": "сумма tokenizer.encode по сырому выводу модели по файлам",
    },
    "Доп. Комментарии": comments,
    "Скорость": {
        "mean_elapsed_sec_per_image": mean_elapsed,
        "output_dir": str(OUTPUT_DIR),
    },
}

summary_path = OUTPUT_DIR / "got_metrics_summary.json"
summary_path.write_text(json.dumps(summary_row, ensure_ascii=False, indent=2), encoding="utf-8")

sum_df = pd.DataFrame([summary_row])
display(HTML("<h3>Сводная строка (экспорт в JSON)</h3>"))
display(HTML(f"<p>Сохранено: <code>{summary_path}</code></p>"))

# Плоское отображение для копирования в таблицу
flat = {
    "Model": summary_row["Model"],
    "Final Score": summary_row["Final Score"],
    "Accuracy": summary_row["Accuracy"],
    "CER": summary_row["CER"],
    "Unit Test Rate": summary_row["Unit Test Rate"],
    "Внутренний парсер": summary_row["Внутренний парсер"],
    "Токены (sum)": summary_row["Токены"]["output_tokens_sum_hypothesis"],
    "Доп. Комментарии": summary_row["Доп. Комментарии"],
}
display(pd.DataFrame([flat]).style.hide(axis="index"))

csv_per_file = OUTPUT_DIR / "got_per_file.csv"
if not df.empty:
    df.to_csv(csv_per_file, index=False)
    print("CSV по файлам:", csv_per_file)

Model,Final Score,Accuracy,CER,Unit Test Rate,Внутренний парсер,Токены (sum),Доп. Комментарии
ucaslcl/GOT-OCR2_0 | chat ocr_type=ocr | low_cpu_mem_usage device_map=cuda(if cuda),100.000000,1.000000,0.000000,1.000000,none,3326,"model=ucaslcl/GOT-OCR2_0 ocr_type=ocr; device=cuda; dtype=torch.float16; n_images=4; with_ref_for_cer=4; draft_ref_txt_written=0 (без эталона до прогона: CER не считался, записан черновик .ref.txt); mean_elapsed_s=33.557125; CER_PASS_THRESHOLD=0.05; normalize=nfkc_ws; expectations_file=ocr_unit_expectations.json"


CSV по файлам: /content/OCR-Analyze/output/got_ocr_notebook/got_per_file.csv
